# Color Palette Analyzer — Workflow en Python

Analiza los colores principales de una imagen en **Google Drive**, usando **Azure AI Foundry**.
Genera un `.zip` con:
- `palette_chart.png` — paleta visual de los 5 colores
- `analisis.txt` — análisis del director de arte
- `analisis_audio.mp3` — audio del análisis (ElevenLabs)


## 1) Instalación y dependencias

In [45]:
!pip install agent-framework-core
!pip install agent-framework-foundry
!pip install elevenlabs requests pillow

## 2) Configuración de credenciales

Rellena las variables directamente aquí.

In [46]:
# ── Azure AI Foundry ──────────────────────────────────────────────
FOUNDRY_ENDPOINT  = "https://n8nprueba-resource.services.ai.azure.com/"              # https://<tu-proyecto>.services.ai.azure.com/
FOUNDRY_API_KEY   = "Fhyf30hJicRBBXThBvTBLNfdNtxco39E3ld4ByG9h8VYM1RJCoMBJQQJ99CFACfhMk5XJ3w3AAAAACOGJMTZ"              # tu API key de Azure AI Foundry
FOUNDRY_MODEL     = "gpt-4o-mini"   # nombre del modelo en tu proyecto Foundry

# ── ElevenLabs ───────────────────────────────────────────────────
ELEVENLABS_API_KEY  = "7469f8785d47cbd4daeb9b8d722316f9542c79aef026cb016c132c9abb8cf7c2"                       # tu API key de ElevenLabs
ELEVENLABS_VOICE_ID = "IKne3meq5aSn9XLyUdCD"  # Charlie

# ── Validación ───────────────────────────────────────────────────
assert FOUNDRY_ENDPOINT, "❌ Falta FOUNDRY_ENDPOINT"
assert FOUNDRY_API_KEY,  "❌ Falta FOUNDRY_API_KEY"

print("✅ Credenciales cargadas")

✅ Credenciales cargadas


## 3) Montar Google Drive

Monta tu Drive para acceder a la imagen. Cuando ejecutes esta celda
Colab te pedirá permiso — acepta y ya tendrás acceso a todos tus archivos
bajo `/content/drive/MyDrive/`.

In [47]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive montado en /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive montado en /content/drive


## 4) Cliente Foundry y AI Agent

Usamos `FoundryChatClient` + `AzureKeyCredential` (compatible con Colab).
El `Agent` se crea una sola vez y se reutiliza en todo el workflow.

In [48]:
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.core.credentials import AccessToken
import time

# Wrapper que satisface la interfaz TokenCredential de Azure
# usando la API key directamente — necesario en Colab (sin Azure CLI)
class ApiKeyCredential:
    def __init__(self, api_key: str):
        self._api_key = api_key

    def get_token(self, *scopes, **kwargs) -> AccessToken:
        # Devuelve la key como token con expiración lejana
        return AccessToken(self._api_key, int(time.time()) + 3600)


SYSTEM_PROMPT = (
    "Actúa como un director de arte experto en colorimetría. "
    "Analiza la imagen y extrae los 5 colores principales. "
    "Después, evalúa la paleta actual y sugiere 2 paletas alternativas "
    "explicando por qué mejorarían el diseño.\n"
    "Devuelve ÚNICAMENTE un objeto JSON con esta estructura exacta, sin texto fuera del JSON:\n"
    "{\n"
    '"colores": [\n'
    '{"hex": "E3C5A8", "codigo": "14-1217 TCX", "nombre": "Amberlight"},\n'
    '{"hex": "B5C7D6", "codigo": "14-4112 TCX", "nombre": "Skyway"}\n'
    "],\n"
    '"analisis": "Tu opinión de la paleta original y tus 2 nuevas propuestas."\n'
    "}\n"
    "Regla: Extrae entre 3 y 6 colores según los que realmente aparezcan en la imagen. Los hex no deben llevar la almohadilla (#)."
)

_foundry_client = FoundryChatClient(
    project_endpoint=FOUNDRY_ENDPOINT,
    model=FOUNDRY_MODEL,
    credential=ApiKeyCredential(FOUNDRY_API_KEY),
)

color_agent = Agent(
    client=_foundry_client,
    name="ColorPaletteAgent",
    instructions=SYSTEM_PROMPT,
)

print("✅ FoundryChatClient y Agent listos")

✅ FoundryChatClient y Agent listos


## 5) AI Agent — análisis de imagen

Envía la imagen en base64 al agente Foundry y parsea el JSON de respuesta.

In [49]:
import base64
import json
import re
from pathlib import Path
from openai import AzureOpenAI


def load_image_as_base64(image_path: str) -> tuple[str, str]:
    """Carga una imagen local y devuelve (base64_data, media_type)."""
    ext = Path(image_path).suffix.lower()
    media_type_map = {
        ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
        ".png": "image/png",  ".gif": "image/gif", ".webp": "image/webp",
    }
    media_type = media_type_map.get(ext, "image/jpeg")
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8"), media_type


async def analyze_image(image_path: str) -> dict:
    """
    Envía la imagen al modelo usando la API de visión directamente
    con el formato multimodal correcto (content array con image_url).
    """
    b64_data, media_type = load_image_as_base64(image_path)

    # Llamada directa con AzureOpenAI para garantizar visión multimodal
    client = AzureOpenAI(
        api_key=FOUNDRY_API_KEY,
        azure_endpoint=FOUNDRY_ENDPOINT,
        api_version="2024-02-15-preview",
    )

    response = client.chat.completions.create(
        model=FOUNDRY_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{media_type};base64,{b64_data}"
                        },
                    },
                    {
                        "type": "text",
                        "text": "Analiza esta imagen y devuelve el JSON."
                    },
                ],
            },
        ],
        max_tokens=1000,
    )

    ai_text = response.choices[0].message.content
    ai_text_clean = re.sub(r"```json|```", "", ai_text).strip()
    return json.loads(ai_text_clean)


print("✅ Función analyze_image lista")

✅ Función analyze_image lista


## 6) Build Chart URL (nodo `Code in JavaScript`)

Réplica exacta del nodo JS: construye la URL de QuickChart con las barras
de colores y etiquetas blancas rotadas.

In [50]:
import urllib.parse


def build_chart_url(colores: list[dict]) -> str:
    """
    Equivalente al nodo 'Code in JavaScript' de n8n.
    Genera la URL de QuickChart con la paleta de 5 colores.
    """
    hex_codes = ["#" + c["hex"].lstrip("#") for c in colores]
    labels    = [["ABB", c["codigo"]] for c in colores]  # 2 líneas por barra, igual que el JS

    chart_config = {
        "type": "bar",
        "data": {
            "labels": labels,
            "datasets": [{
                "data": [1, 1, 1, 1, 1],
                "backgroundColor": hex_codes,
                "barPercentage": 1.0,
                "categoryPercentage": 1.0,
            }],
        },
        "options": {
            "layout": {"padding": 0},
            "legend": {"display": False},
            "scales": {
                "xAxes": [{"display": False}],
                "yAxes": [{"display": False, "ticks": {"min": 0, "max": 1}}],
            },
            "plugins": {
                "datalabels": {
                    "color": "#000000",
                    "backgroundColor": "#ffffff",
                    "borderRadius": 4,
                    "padding": 12,
                    "rotation": 90,
                    "anchor": "end",
                    "align": "start",
                    "offset": 40,
                    "font": {"family": "sans-serif", "size": 16, "weight": "bold"},
                    "formatter": "INYECCION_DE_CODIGO",
                },
            },
        },
    }

    # Inyección de función JS — réplica exacta del replace del nodo Code de n8n
    config_str = json.dumps(chart_config)
    config_str = config_str.replace(
        '"INYECCION_DE_CODIGO"',
        "function(value, context) { return context.chart.data.labels[context.dataIndex]; }",
    )

    return "https://quickchart.io/chart?w=800&h=600&bkg=white&c=" + urllib.parse.quote(config_str)


print("✅ Función build_chart_url lista")

✅ Función build_chart_url lista


## 7) Download Chart Image (`HTTP Request`)

In [51]:
import requests


def download_chart_image(chart_url: str, output_path: str = "palette_chart.png") -> str:
    """
    Equivalente al nodo 'HTTP Request' de n8n (responseFormat: file).
    Descarga la imagen de QuickChart y la guarda en disco.
    """
    response = requests.get(chart_url, timeout=30)
    response.raise_for_status()
    Path(output_path).write_bytes(response.content)
    print(f"📊 Paleta guardada en: {output_path}")
    return output_path


print("✅ Función download_chart_image lista")

✅ Función download_chart_image lista


## 8) Text-to-Speech con ElevenLabs (nodo `Convert text to speech`)

Usa la voz `Charlie` (IKne3meq5aSn9XLyUdCD).

In [52]:
from elevenlabs.client import ElevenLabs


def elevenlabs_tts(text: str, output_path: str = "analisis_audio.mp3") -> str:
    """
    Equivalente al nodo 'Convert text to speech' de n8n.
    Convierte el texto de análisis a audio con la voz Charlie de ElevenLabs.
    """
    el_client = ElevenLabs(api_key=ELEVENLABS_API_KEY)

    audio_generator = el_client.text_to_speech.convert(
        voice_id=ELEVENLABS_VOICE_ID,
        text=text,
        model_id="eleven_multilingual_v2",
    )

    audio_bytes = b"".join(
        chunk if isinstance(chunk, bytes) else bytes(chunk)
        for chunk in audio_generator
    )

    Path(output_path).write_bytes(audio_bytes)
    print(f"🔊 Audio guardado en: {output_path}")
    return output_path
"Regla: Extrae entre 3 y 6 colores según los que realmente aparezcan en la imagen. Los hex no deben llevar la almohadilla (#)."

print("✅ Función elevenlabs_tts lista")

✅ Función elevenlabs_tts lista


## 9) Empaquetar resultados en ZIP

Empaqueta la paleta PNG, el análisis TXT y el audio MP3 en un único `.zip`
que Colab descarga automáticamente.

In [53]:
import shutil
import tempfile
import zipfile
from pathlib import Path


def build_zip(
    chart_path: str,
    analisis: str,
    audio_path: str | None,
    zip_path: str = "/content/resultado_paleta.zip",
) -> str:
    """
    Empaqueta los 3 entregables en un ZIP y lo descarga desde Colab.
      - palette_chart.png
      - analisis.txt
      - analisis_audio.mp3  (solo si se generó audio)
    """
    with tempfile.TemporaryDirectory() as tmpdir:
        tmp = Path(tmpdir)

        shutil.copy(chart_path, tmp / "palette_chart.png")
        (tmp / "analisis.txt").write_text(analisis, encoding="utf-8")
        if audio_path and Path(audio_path).exists():
            shutil.copy(audio_path, tmp / "analisis_audio.mp3")

        zip_base = zip_path.removesuffix(".zip")
        shutil.make_archive(zip_base, "zip", root_dir=tmpdir)

    final = zip_base + ".zip"

    # Mostrar contenido
    print("\n📦 Contenido del ZIP:")
    with zipfile.ZipFile(final) as zf:
        for name in zf.namelist():
            size = zf.getinfo(name).file_size
            print(f"   {name:30s}  {size / 1024:.1f} KB")

    # Descarga automática en Colab
    from google.colab import files
    files.download(final)
    print(f"\n✅ ZIP descargado: {final}")
    return final


print("✅ Función build_zip lista")

✅ Función build_zip lista


## 10) Workflow completo (`@workflow`)

In [54]:
from agent_framework import workflow


@workflow
async def color_palette_workflow(request: dict) -> dict:
    """
    Workflow completo.
    Claves del dict de entrada:
      - image_path     (str)  ruta a la imagen en Drive
      - generate_audio (bool) activar ElevenLabs, default True
      - zip_output     (str)  ruta del ZIP de salida
    """
    image_path     = request["image_path"]
    generate_audio = request.get("generate_audio", True)
    zip_output     = request.get("zip_output", "/content/resultado_paleta.zip")

    print(f"\n🚀 Iniciando workflow | imagen: {image_path}")

    # ── AI Agent ──────────────────────────────────────────────────
    print("\n🤖 [AI Agent] Analizando imagen con Azure AI Foundry...")
    ai_data  = await analyze_image(image_path)
    colores  = ai_data["colores"]
    analisis = ai_data["analisis"]
    print(f"   Colores extraídos: {[c['nombre'] for c in colores]}")

    # ── QuickChart ────────────────────────────────────────────────
    print("\n🎨 [Code JS] Construyendo URL de QuickChart...")
    chart_url = build_chart_url(colores)

    # ── Descargar paleta ──────────────────────────────────────────
    print("\n📥 [HTTP Request] Descargando imagen de paleta...")
    chart_path = download_chart_image(chart_url, output_path="/content/palette_chart.png")
    print(f"   Guardado en: {chart_path}")

    # ── TTS ElevenLabs ────────────────────────────────────────────
    audio_path = None
    if generate_audio:
        print("\n🎙️ [ElevenLabs] Generando audio del análisis...")
        audio_path = elevenlabs_tts(analisis, output_path="/content/analisis_audio.mp3")
        print(f"   Guardado en: {audio_path}")

    # ── ZIP ───────────────────────────────────────────────────────
    print("\n📦 Empaquetando resultados...")
    build_zip(chart_path, analisis, audio_path, zip_path=zip_output)

    print("\n✅ Workflow completado")
    return {"colores": colores, "analisis": analisis,
            "chart_path": chart_path, "audio_path": audio_path}


print("✅ Workflow definido")

✅ Workflow definido


## 11) Ejecución

Se añade la ruta de donde se extrae la foto y donde se guradará el zip con el contenido.

In [55]:
IMAGE_PATH = "/content/drive/MyDrive/fotoanalisis.png"

result = await color_palette_workflow.run({
    "image_path":     IMAGE_PATH,
    "generate_audio": True,           # ← False si no tienes clave de ElevenLabs
    "zip_output":     "/content/resultado_paleta.zip",
})

resultado = result.get_outputs()[0]


🚀 Iniciando workflow | imagen: /content/drive/MyDrive/fotoanalisis.png

🤖 [AI Agent] Analizando imagen con Azure AI Foundry...
   Colores extraídos: ['Turquoise', 'Blue', 'Light Beige', 'Light Green']

🎨 [Code JS] Construyendo URL de QuickChart...

📥 [HTTP Request] Descargando imagen de paleta...
📊 Paleta guardada en: /content/palette_chart.png
   Guardado en: /content/palette_chart.png

🎙️ [ElevenLabs] Generando audio del análisis...
🔊 Audio guardado en: /content/analisis_audio.mp3
   Guardado en: /content/analisis_audio.mp3

📦 Empaquetando resultados...

📦 Contenido del ZIP:
   palette_chart.png               29.5 KB
   analisis.txt                    0.4 KB
   analisis_audio.mp3              391.5 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ ZIP descargado: /content/resultado_paleta.zip

✅ Workflow completado
